# Project SQL: Data Discovery with the Chinook Database

Welcome to your first major hands-on data exploration project. In this assignment, you will be stepping into the role of a data analyst querying the Chinook database: a sample dataset representing a digital media store, complete with information about artists, albums, tracks, customers, and business sales.

The core objective of this project is to transition you from simply writing code to thinking like a data scientist. To do that, our journey follows a deliberate, four-fold progression:



### Phase 1. Master the Mechanics (Get Familiar with SQL)



Before you can extract deep insights, you need to speak the language. You will start by practicing the fundamental mechanics of SQL using the SQL statements below: learning how to filter data, join tables, group metrics, and aggregate numbers. Think of this as learning how to move around the data environment smoothly.  As a starting point, pick a single table and answer the "How many ...?" questions using SQL.  Then repeat the process for each table.



In [4]:
# Install the sqlite package for Ubuntu
# Download the Chinook sqlite database


In [5]:
#python import
import sqlite3 as db
import pandas as pd
import contextlib

In [6]:
# Install the sqlite package for Ubuntu
%%capture
%%bash
apt-get update
apt-get install -y sqlite3

In [7]:
# Download the Chinook sqlite database
%%bash
[ -f chinook.zip ] || curl -s -O https://www.sqlitetutorial.net/wp-content/uploads/2018/03/chinook.zip
unzip -o chinook.zip


Archive:  chinook.zip
  inflating: chinook.db              


In [8]:
#Remove any old copy, then unzip
!rm -f chinook.db
!unzip -u chinook.zip

Archive:  chinook.zip
  inflating: chinook.db              


In [9]:
# Check the files
!ls -l

total 1168
-rw-r--r-- 1 root root 884736 Nov 29  2015 chinook.db
-rw-r--r-- 1 root root 305596 Nov 13 14:49 chinook.zip
drwxr-xr-x 1 root root   4096 Nov 11 15:59 sample_data


### Run sql using jupyter magic

#### Count how many columns in each table

In [10]:
# List all the tables in the Chinook database
%%script sqlite3 --column --header chinook.db
.tables

albums          employees       invoices        playlists     
artists         genres          media_types     tracks        
customers       invoice_items   playlist_track


In [11]:
# Show the columns in the albums table
%%script sqlite3 --column --header chinook.db
pragma table_info(albums);

cid  name      type           notnull  dflt_value  pk
---  --------  -------------  -------  ----------  --
0    AlbumId   INTEGER        1                    1 
1    Title     NVARCHAR(160)  1                    0 
2    ArtistId  INTEGER        1                    0 


In [12]:
# How many columns are in the albums table?
%%script sqlite3 --column --header chinook.db
select count(*) as n_columns.
from pragma_table_info('albums')


Parse error near line 1: near ".": syntax error
  select count(*) as n_columns.        from pragma_table_info('albums')  
                error here ---^


CalledProcessError: Command 'b"select count(*) as n_columns.       \nfrom pragma_table_info('albums')  \n"' returned non-zero exit status 1.

In [ ]:
# How many columns are in each table?
%%script sqlite3 --column --header chinook.db
select 'albums'         as table_name, count(*) as n_columns from pragma_table_info('albums');
select 'artists'        as table_name, count(*) as n_columns from pragma_table_info('artists');
select 'customers'      as table_name, count(*) as n_columns from pragma_table_info('customers');
select 'employees'      as table_name, count(*) as n_columns from pragma_table_info('employees');
select 'genres'         as table_name, count(*) as n_columns from pragma_table_info('genres');
select 'invoice_items'  as table_name, count(*) as n_columns from pragma_table_info('invoice_items');
select 'invoices'       as table_name, count(*) as n_columns from pragma_table_info('invoices');
select 'media_types'    as table_name, count(*) as n_columns from pragma_table_info('media_types');
select 'playlist_track' as table_name, count(*) as n_columns from pragma_table_info('playlist_track');
select 'playlists'      as table_name, count(*) as n_columns from pragma_table_info('playlists');
select 'tracks'         as table_name, count(*) as n_columns from pragma_table_info('tracks');

#### Count how many rows in each table

In [ ]:
# How many rows are in the albums table?
%%script sqlite3 --column --header chinook.db
select count(*) as n_rows
from albums;      # look at the table of rows

In [ ]:
# How many rows are in each table?
%%script sqlite3 --column --header chinook.db
select 'albums'         as table_name, count(*) as n_rows from albums;
select 'artists'        as table_name, count(*) as n_rows from artists;
select 'customers'      as table_name, count(*) as n_rows from customers;
select 'employees'      as table_name, count(*) as n_rows from employees;
select 'genres'         as table_name, count(*) as n_rows from genres;
select 'invoice_items'  as table_name, count(*) as n_rows from invoice_items;
select 'invoices'       as table_name, count(*) as n_rows from invoices;
select 'media_types'    as table_name, count(*) as n_rows from media_types;
select 'playlist_track' as table_name, count(*) as n_rows from playlist_track;
select 'playlists'      as table_name, count(*) as n_rows from playlists;
select 'tracks'         as table_name, count(*) as n_rows from tracks;

#### Break down columns

In [ ]:
# Identifier columns are the Primary keys and Foreign keys in each table
# Which column is the Primary key in "tracks"?
%%script sqlite3 --column --header chinook.db
select name, pk
from pragma_table_info('tracks')
where pk > 0;


In [ ]:
# Which columns in track tablet is the (Foreign key) point to other tables?
%%script sqlite3 --column --header chinook.db
select "from" as column_name, "table" as points_to
from pragma_foreign_key_list('tracks');

In [ ]:
# Is TrackId unique? (the two numbers should match)
%%script sqlite3 --column --header chinook.db
select count(*)                as n_rows,
       count(distinct TrackId) as n_unique_ids
from tracks;

### Run SQl using DuckDB

In [13]:
import duckdb
import pandas as pd

con = duckdb.connect()                                  # an empty DuckDB session in memory
con.sql("INSTALL sqlite; LOAD sqlite;")                 # add-on that lets DuckDB read SQLite files
con.sql("ATTACH 'chinook.db' AS chinook (TYPE sqlite)") # plug in your file and call it 'chinook'
con.sql("USE chinook")                                  # make it the default, so you can write 'albums'

#### Column count

In [14]:
#====================================================
# HOW MANY COLUMN IN EACH TABLE ?
#(note show what change if have diffrent database)
#====================================================

con.sql("""
    select table_name,                          -- never changes
           count(*) as n_columns                -- never changes (the name n_columns is your choice)
    from information_schema.columns             -- never changes
    where table_catalog = 'chinook'             -- CHANGE: your ATTACH nickname
      and table_name not like 'sqlite_%'        -- keep only for SQLite files
    group by table_name                         -- never changes
    order by table_name                         -- never changes
""").df()

,table_name,n_columns
0,albums,3
1,artists,2
2,customers,13
3,employees,15
4,genres,2
5,invoice_items,5
6,invoices,9
7,media_types,2
8,playlist_track,2
9,playlists,2


In [ ]:
con.sql("SHOW TABLES").df()

#### Row count

In [15]:
# ============================================================
# HOW MANY ROWS ARE IN EACH TABLE?
# ------------------------------------------------------------
# Idea: ask "how many rows?" once per table, then stack all
#       the answers into one result table.
#
# TO REUSE ON ANOTHER DATABASE:
#   1. Run con.sql("SHOW TABLES").df() to get the table names.
#   2. Make one select line per table.
#   3. On each line, change the name in TWO places:
#        'label'      -> text shown in the output (in quotes)
#        from table   -> the real table being counted (no quotes)
#   4. Put "union all" BETWEEN lines (not before the first one,
#      not after the last one).
# ============================================================

con.sql("""
    -- Each select line answers: "How many rows are in this table?"
    --   'albums'           = a text label we type, so we know which table the number belongs to
    --   as table_name      = names the label column
    --   count(*)           = counts every row, even rows with blank values
    --   as n_rows          = names the count column
    --   from albums        = the table being counted (must match the label)

    select 'albums'         as table_name, count(*) as n_rows from albums
    union all                                   -- "stack the next answer under this one"
    select 'artists'        as table_name, count(*) as n_rows from artists
    union all
    select 'customers'      as table_name, count(*) as n_rows from customers
    union all
    select 'employees'      as table_name, count(*) as n_rows from employees
    union all
    select 'genres'         as table_name, count(*) as n_rows from genres
    union all
    select 'invoice_items'  as table_name, count(*) as n_rows from invoice_items
    union all
    select 'invoices'       as table_name, count(*) as n_rows from invoices
    union all
    select 'media_types'    as table_name, count(*) as n_rows from media_types
    union all
    select 'playlist_track' as table_name, count(*) as n_rows from playlist_track
    union all
    select 'playlists'      as table_name, count(*) as n_rows from playlists
    union all
    select 'tracks'         as table_name, count(*) as n_rows from tracks

    order by table_name                         -- sort the final stacked table A to Z (written once, at the end)
""").df()                                       # .df() only DISPLAYS the result as a pandas table; SQL did the counting

# EASY MISTAKE: changing the 'label' but not the "from" table.
#   You'd get the wrong number under the right name, and NO error.

,table_name,n_rows
0,albums,347
1,artists,275
2,customers,59
3,employees,8
4,genres,25
5,invoice_items,2240
6,invoices,412
7,media_types,5
8,playlist_track,8715
9,playlists,18


#### Identifier Columns

##### Find primary keys

In [16]:
# ============================================================
# WHICH COLUMNS ARE PRIMARY KEYS?
# ------------------------------------------------------------
# duckdb_constraints() is DuckDB's built-in list of the "rules"
# on each table: primary keys, NOT NULL, and so on.
# We keep only the PRIMARY KEY rules.
#
# TO REUSE: change 'chinook' to your ATTACH nickname.
# ============================================================

con.sql("""
    select table_name,
           unnest(constraint_column_names) as column_name   -- unnest = split a list into one row per item
    from duckdb_constraints()                               -- built-in list of table rules
    where database_name = 'chinook'                         -- only our database
      and constraint_type = 'PRIMARY KEY'                   -- only primary-key rules
    order by table_name
""").df()

,table_name,column_name
0,albums,AlbumId
1,artists,ArtistId
2,customers,CustomerId
3,employees,EmployeeId
4,genres,GenreId
5,invoice_items,InvoiceLineId
6,invoices,InvoiceId
7,media_types,MediaTypeId
8,playlist_track,PlaylistId
9,playlist_track,TrackId


In [ ]:
# ============================================================
# WHICH COLUMNS ARE FOREIGN KEYS (point to another table)?
# ------------------------------------------------------------
# DuckDB does NOT bring SQLite's foreign keys over when it
# attaches the file. So we use sqlite_query() to send a
# question straight to SQLite and get its answer back.
#
# Everything inside the single quotes '...' is SQLite's SQL.
# Inside that, '' (two single quotes) = one quote mark.
#
# TO REUSE: change 'chinook' to your ATTACH nickname.
#           (Only works for SQLite files.)
# ============================================================

con.sql("""
    select table_name, column_name
    from sqlite_query('chinook', '
        select m.name   as table_name,                  -- the table
               f."from" as column_name                  -- the column that points elsewhere
        from sqlite_master as m                          -- SQLite''s list of everything in the file
        join pragma_foreign_key_list(m.name) as f        -- foreign keys for each table
        where m.type = ''table''                         -- tables only (not indexes)
    ')
    order by table_name
""").df()

In [ ]:
# ============================================================
# HOW MANY PRIMARY KEYS, FOREIGN KEYS, AND IDENTIFIER COLUMNS
# ARE IN EACH TABLE?
# ------------------------------------------------------------
# "with" builds temporary named tables that only exist while
# this query runs. Here we build three, step by step:
#   1. primary_keys -> every primary-key column, tagged 'PK'
#   2. foreign_keys -> every foreign-key column, tagged 'FK'
#   3. all_keys     -> both lists stacked (UNION ALL keeps every row,
#                      so we can count PKs and FKs separately)
# Then the final select counts each kind per table.
#
# TO REUSE: change 'chinook' in BOTH places to your ATTACH nickname.
# ============================================================

con.sql("""
    with
    -- Step 1: list every primary-key column, and tag it 'PK'
    primary_keys as (
        select table_name,
               unnest(constraint_column_names) as column_name,  -- split key lists into one row per column
               'PK' as key_type                                 -- a label we add, so we know what kind of key it is
        from duckdb_constraints()                               -- DuckDB's built-in list of table rules
        where database_name = 'chinook'
          and constraint_type = 'PRIMARY KEY'
    ),

    -- Step 2: list every foreign-key column (asked directly from SQLite), and tag it 'FK'
    foreign_keys as (
        select table_name, column_name, 'FK' as key_type
        from sqlite_query('chinook', '
            select m.name as table_name, f."from" as column_name
            from sqlite_master as m
            join pragma_foreign_key_list(m.name) as f
            where m.type = ''table''
        ')
    ),

    -- Step 3: stack both lists, keeping every row
    all_keys as (
        select * from primary_keys
        union all
        select * from foreign_keys
    )

    -- Final answer: count each kind of key per table
    select table_name,
           count(*) filter (where key_type = 'PK') as n_pk,                  -- count only the PK rows
           count(*) filter (where key_type = 'FK') as n_fk,                  -- count only the FK rows
           count(distinct column_name)            as n_identifier_columns   -- count each column once, even if it's both
    from all_keys
    group by table_name
    order by table_name
""").df()

In [ ]:
# ============================================================
# HOW MANY FEATURE COLUMNS ARE IN EACH TABLE?
# ------------------------------------------------------------
# Feature columns = all columns that are NOT identifier columns.
# (No target chosen yet. When you pick one, subtract 1 more.)
#
# "with" builds temporary named tables for this query only:
#   1. all_columns        -> every column in every table
#   2. primary_keys       -> every primary-key column
#   3. foreign_keys       -> every foreign-key column
#   4. identifier_columns -> PK + FK combined, repeats removed
# Then we match all_columns against identifier_columns.
# Whatever does NOT match is a feature column.
#
# TO REUSE: change 'chinook' in all THREE places to your ATTACH nickname.
# ============================================================

con.sql("""
    with
    -- Step 1: every column in every table
    all_columns as (
        select table_name, column_name
        from information_schema.columns                 -- one row per column
        where table_catalog = 'chinook'
          and table_name not like 'sqlite_%'
    ),

    -- Step 2: every primary-key column
    primary_keys as (
        select table_name,
               unnest(constraint_column_names) as column_name
        from duckdb_constraints()
        where database_name = 'chinook'
          and constraint_type = 'PRIMARY KEY'
    ),

    -- Step 3: every foreign-key column (asked directly from SQLite)
    foreign_keys as (
        select table_name, column_name
        from sqlite_query('chinook', '
            select m.name as table_name, f."from" as column_name
            from sqlite_master as m
            join pragma_foreign_key_list(m.name) as f
            where m.type = ''table''
        ')
    ),

    -- Step 4: PK + FK combined, repeats removed
    identifier_columns as (
        select table_name, column_name from primary_keys
        union
        select table_name, column_name from foreign_keys
    )

    -- Final answer: match every column against the identifier list
    select a.table_name,
           count(*)                        as n_columns,             -- all columns
           count(i.column_name)            as n_identifier_columns,  -- the ones that matched
           count(*) - count(i.column_name) as n_feature_columns      -- the ones that did NOT match
    from all_columns as a
    left join identifier_columns as i                   -- keep EVERY column, matched or not
           on  a.table_name  = i.table_name             -- same table...
           and a.column_name = i.column_name            -- ...and same column name
    group by a.table_name
    order by a.table_name
""").df()

In [ ]:
# ============================================================
# WHAT DATA TYPES ARE IN THIS DATABASE?
# ------------------------------------------------------------
# Run this first on ANY new database, so you know which type
# names to treat as "numbers" in the next query.
# TO REUSE: change 'chinook' to your ATTACH nickname.
# ============================================================

con.sql("""
    select data_type,                               -- the kind of value: number, text, date...
           count(*) as n_columns                    -- how many columns have this type
    from information_schema.columns
    where table_catalog = 'chinook'
      and table_name not like 'sqlite_%'
    group by data_type
    order by n_columns desc
""").df()

In [ ]:
# ============================================================
# HOW MANY NUMERICAL COLUMNS ARE IN EACH TABLE?
# ------------------------------------------------------------
# Numerical column = a FEATURE column (not an ID) whose
#                    data type is a number type.
#
# Same steps as the feature-column query, plus ONE new thing:
#   is_number -> a true/false flag we add in Step 1
#                (true if the data type is a number type)
#
# TO REUSE: change 'chinook' in all THREE places.
#           If your database has other number types, add them
#           to the is_number list (check with the data-type query).
# ============================================================

con.sql("""
    with
    -- Step 1: every column, plus a true/false flag: is it a number type?
    all_columns as (
        select table_name,
               column_name,
               data_type,
               data_type in ('TINYINT', 'SMALLINT', 'INTEGER', 'BIGINT',
                             'FLOAT', 'DOUBLE', 'DECIMAL') as is_number   -- true or false
        from information_schema.columns
        where table_catalog = 'chinook'
          and table_name not like 'sqlite_%'
    ),

    -- Step 2: every primary-key column
    primary_keys as (
        select table_name,
               unnest(constraint_column_names) as column_name
        from duckdb_constraints()
        where database_name = 'chinook'
          and constraint_type = 'PRIMARY KEY'
    ),

    -- Step 3: every foreign-key column (asked directly from SQLite)
    foreign_keys as (
        select table_name, column_name
        from sqlite_query('chinook', '
            select m.name as table_name, f."from" as column_name
            from sqlite_master as m
            join pragma_foreign_key_list(m.name) as f
            where m.type = ''table''
        ')
    ),

    -- Step 4: PK + FK combined, repeats removed
    identifier_columns as (
        select table_name, column_name from primary_keys
        union
        select table_name, column_name from foreign_keys
    )

    -- Final answer
    --   i.column_name is null  = no match in the ID list = a feature column
    --   a.is_number            = the column holds numbers
    select a.table_name,
           count(*) filter (where i.column_name is null)                 as n_feature_columns,
           count(*) filter (where i.column_name is null and a.is_number) as n_numerical_columns,
           string_agg(a.column_name, ', ')                                         -- glue the names into one text line
                    filter (where i.column_name is null and a.is_number) as numerical_columns
    from all_columns as a
    left join identifier_columns as i
           on  a.table_name  = i.table_name
           and a.column_name = i.column_name
    group by a.table_name
    order by a.table_name
""").df()

In [ ]:
# ============================================================
# HOW MANY CATEGORICAL COLUMNS ARE IN EACH TABLE?
# ------------------------------------------------------------
# Categorical column = a FEATURE column (not an ID) stored as text.
#
# Same steps as the numerical query, with two more flags:
#   is_number   -> true if the type is a number type
#   is_category -> true if the type is text (labels)
#   is_date     -> true if the type is a date/time
# Every feature column should land in exactly ONE of these.
# CHECK: numerical + categorical + date = feature columns.
#
# TO REUSE: change 'chinook' in all THREE places.
#           Run the data-type query first; add any new type
#           names to the right flag list.
# ============================================================

con.sql("""
    with
    -- Step 1: every column, plus three true/false flags for its kind
    all_columns as (
        select table_name,
               column_name,
               data_type,
               data_type in ('TINYINT', 'SMALLINT', 'INTEGER', 'BIGINT',
                             'FLOAT', 'DOUBLE', 'DECIMAL')   as is_number,
               data_type in ('VARCHAR', 'TEXT', 'BOOLEAN')   as is_category,
               data_type in ('DATE', 'TIMESTAMP', 'TIME')    as is_date
        from information_schema.columns
        where table_catalog = 'chinook'
          and table_name not like 'sqlite_%'
    ),

    -- Step 2: every primary-key column
    primary_keys as (
        select table_name,
               unnest(constraint_column_names) as column_name
        from duckdb_constraints()
        where database_name = 'chinook'
          and constraint_type = 'PRIMARY KEY'
    ),

    -- Step 3: every foreign-key column (asked directly from SQLite)
    foreign_keys as (
        select table_name, column_name
        from sqlite_query('chinook', '
            select m.name as table_name, f."from" as column_name
            from sqlite_master as m
            join pragma_foreign_key_list(m.name) as f
            where m.type = ''table''
        ')
    ),

    -- Step 4: PK + FK combined, repeats removed
    identifier_columns as (
        select table_name, column_name from primary_keys
        union
        select table_name, column_name from foreign_keys
    )

    -- Final answer
    --   i.column_name is null = not an ID = a feature column
    select a.table_name,
           count(*) filter (where i.column_name is null)                   as n_feature_columns,
           count(*) filter (where i.column_name is null and a.is_number)   as n_numerical_columns,
           count(*) filter (where i.column_name is null and a.is_category) as n_categorical_columns,
           count(*) filter (where i.column_name is null and a.is_date)     as n_date_columns,
           string_agg(a.column_name, ', ')                                           -- list the names so you can see them
                    filter (where i.column_name is null and a.is_category) as categorical_columns
    from all_columns as a
    left join identifier_columns as i
           on  a.table_name  = i.table_name
           and a.column_name = i.column_name
    group by a.table_name
    order by a.table_name
""").df()

In [ ]:
# ============================================================
# HOW MANY NESTED COLUMNS ARE IN EACH TABLE?
# ------------------------------------------------------------
# Nested column = one cell holds several values.
# In DuckDB, nested types look like:
#   INTEGER[]  or  VARCHAR[]   -> a LIST    (ends in [])
#   STRUCT(...)                -> a record  (several named fields)
#   MAP(...)                   -> key/value pairs
#   JSON                       -> JSON text
#
# TO REUSE: change 'chinook' to your ATTACH nickname.
# ============================================================

con.sql("""
    select table_name,
           count(*) filter (where data_type like '%[]'          -- lists end in []
                               or data_type like 'STRUCT%'      -- records start with STRUCT
                               or data_type like 'MAP%'         -- maps start with MAP
                               or data_type = 'JSON')           -- JSON
               as n_nested_columns
    from information_schema.columns
    where table_catalog = 'chinook'
      and table_name not like 'sqlite_%'
    group by table_name
    order by table_name
""").df()